In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

encoder_epoch_df = pd.read_hdf(
    Path(pdata_root) / "_cache" / "behavior_epoch_metrics.h5",
    key="encoder/prepost_1s_speedThresh_1cms"
)

from src.qc.qc_events import load_behavior_qc_tables

events_df, session_summary_df = load_behavior_qc_tables(
    pdata_root=pdata_root,
    filename="behavior_QC.h5"
)

from src.utils.pdata_organize import make_session_availability_summary

session_availability_df = make_session_availability_summary(
    events_df=events_df,
    windows_df=windows_df,
    min_session_duration_s=900,
    min_valid_events=3,
)

from src.utils.pdata_organize import add_day_bins_to_sessions

session_day_df = add_day_bins_to_sessions(
    session_availability_df,
    use_good_sessions_only=True,
)

Current working directory: /home/nmldata2/ccaw/Python/notebooks
Added repo root: /home/nmldata2/ccaw/Python
src exists: True
/home/nmldata2/ccaw/Python
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s
All windows: (103980, 40)
Valid windows: (101861, 40)
[LOADED] Behavior QC tables: /mnt/pdata/Classical_Conditioning/_cache/behavior_QC.h5


In [2]:
# --------------------------------------------------
# Inspect phase and anchor names
# --------------------------------------------------

print("Phases:")
print(encoder_epoch_df["phase"].value_counts())

anchor_col = "anchor_name" if "anchor_name" in encoder_epoch_df.columns else "epoch_name"
print("\nUsing anchor column:", anchor_col)

print("\nAnchors by phase:")
display(
    encoder_epoch_df
    .groupby(["phase", anchor_col])
    .size()
    .reset_index(name="n")
    .sort_values(["phase", anchor_col])
)

print("\nWindow positions:")
print(encoder_epoch_df["window_position"].value_counts())

Phases:
phase
air_training         42362
habituation          38237
tone_air_training    21262
Name: count, dtype: int64

Using anchor column: anchor_name

Anchors by phase:


,phase,anchor_name,n
0,air_training,air_off,7060
1,air_training,air_off_mid,7033
2,air_training,air_on,7067
3,air_training,air_on_mid,7069
4,air_training,pseudo_tone_off,7070
5,air_training,pseudo_tone_on,7063
6,habituation,LED_off,6373
7,habituation,LED_off_mid,6327
8,habituation,LED_on,6390
9,habituation,LED_on_mid,6386



Window positions:
window_position
pre     50952
post    50909
Name: count, dtype: int64


In [3]:
# --------------------------------------------------
# Choose tone-air phase
# --------------------------------------------------

tone_phase_candidates = [
    p for p in encoder_epoch_df["phase"].dropna().unique()
    if "tone" in str(p).lower()
]

print("Tone phase candidates:", tone_phase_candidates)

tone_air_phase = tone_phase_candidates[0]
print("Using tone-air phase:", tone_air_phase)

Tone phase candidates: ['tone_air_training']
Using tone-air phase: tone_air_training


In [4]:
# --------------------------------------------------
# Prepare tone-air window dataframe
# --------------------------------------------------

tone_air_anchors = [
    "tone_on",
    "air_on",
    "tone_off",
    "air_on_mid",
    "air_off",
    "air_off_mid",
]

tone_air_epoch_map = {
    ("tone_on", "pre"): "pre_tone_on",
    ("tone_on", "post"): "post_tone_on",

    ("air_on", "pre"): "pre_air_on",
    ("air_on", "post"): "post_air_on",

    ("tone_off", "pre"): "pre_tone_off",
    ("tone_off", "post"): "post_tone_off",

    ("air_on_mid", "pre"): "pre_air_on_mid",
    ("air_on_mid", "post"): "post_air_on_mid",

    ("air_off", "pre"): "pre_air_off",
    ("air_off", "post"): "post_air_off",

    ("air_off_mid", "pre"): "pre_air_off_mid",
    ("air_off_mid", "post"): "post_air_off_mid",
}

tone_air_cycle_order = [
    "pre_tone_on",
    "post_tone_on",
    "pre_air_on",
    "post_air_on",
    "pre_tone_off",
    "post_tone_off",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_tone_on_next",
]


def prepare_tone_air_window_df(
    encoder_epoch_df,
    phase,
    anchors,
    anchor_col="anchor_name",
    keep_overlap=True,
    require_good_session=True,
    require_valid_window=True,
):
    """
    Prepare long-format tone-air training window dataframe.

    One row = one pre/post window around a tone-air anchor.
    """

    df = encoder_epoch_df.copy()

    df = df[
        (df["phase"] == phase) &
        (df[anchor_col].isin(anchors)) &
        (df["window_position"].isin(["pre", "post"]))
    ].copy()

    if require_good_session and "good_session_basic" in df.columns:
        df = df[df["good_session_basic"] == True].copy()

    if require_valid_window and "valid_window" in df.columns:
        df = df[df["valid_window"] == True].copy()

    if not keep_overlap and "overlap_flag" in df.columns:
        df = df[df["overlap_flag"] == False].copy()

    # biological epoch labels
    df["tone_air_epoch_label"] = [
        tone_air_epoch_map.get((a, w), np.nan)
        for a, w in zip(df[anchor_col], df["window_position"])
    ]

    df = df[df["tone_air_epoch_label"].notna()].copy()

    # IDs
    df["animal_day"] = (
        df["animal"].astype(str) + ":" +
        df["date"].astype(str)
    )

    # Session time
    if "session_time_min" not in df.columns:
        if "anchor_session_time_min" in df.columns:
            df["session_time_min"] = df["anchor_session_time_min"]
        elif "anchor_time_s" in df.columns:
            df["session_time_min"] = df["anchor_time_s"] / 60.0
        elif "session_time_s" in df.columns:
            df["session_time_min"] = df["session_time_s"] / 60.0
        else:
            df["session_time_min"] = np.nan

    # Exposure session number
    if "phase_session_number" in df.columns:
        df["tone_air_session"] = df["phase_session_number"]
    elif "phase_day_number_good" in df.columns:
        df["tone_air_session"] = df["phase_day_number_good"]
    else:
        # Try merging from session_day_df
        merge_cols = [
            c for c in [
                "animal",
                "date",
                "phase",
                "phase_day_number_good",
                "phase_session_number",
            ]
            if c in session_day_df.columns
        ]

        sess_tmp = session_day_df[merge_cols].drop_duplicates().copy()

        df = df.merge(
            sess_tmp,
            on=[c for c in ["animal", "date", "phase"] if c in merge_cols],
            how="left",
            suffixes=("", "_sess")
        )

        if "phase_session_number" in df.columns:
            df["tone_air_session"] = df["phase_session_number"]
        elif "phase_day_number_good" in df.columns:
            df["tone_air_session"] = df["phase_day_number_good"]
        else:
            df["tone_air_session"] = np.nan

    return df.reset_index(drop=True)


tone_air_window_df = prepare_tone_air_window_df(
    encoder_epoch_df=encoder_epoch_df,
    phase=tone_air_phase,
    anchors=tone_air_anchors,
    anchor_col=anchor_col,
    keep_overlap=True,
    require_good_session=True,
    require_valid_window=True,
)

print(tone_air_window_df.shape)

display(
    tone_air_window_df["tone_air_epoch_label"]
    .value_counts()
    .reindex([x for x in tone_air_cycle_order if x != "pre_tone_on_next"])
)

(21154, 63)


tone_air_epoch_label
pre_tone_on         1766
post_tone_on        1766
pre_air_on          1766
post_air_on         1766
pre_tone_off        1766
post_tone_off       1765
pre_air_on_mid      1766
post_air_on_mid     1766
pre_air_off         1766
post_air_off        1759
pre_air_off_mid     1751
post_air_off_mid    1751
Name: count, dtype: int64

In [5]:
# --------------------------------------------------
# Build full tone-air cycle dataframe
# --------------------------------------------------

def make_tone_air_cycle_df(
    tone_air_window_df,
    trial_col="event_number",
    require_complete_cycles=True,
):
    """
    Build long-format full tone-air cycle dataframe.

    One cycle = trial N.

    Includes:
        all tone/air epochs from trial N
        plus pre_tone_on from trial N+1 as pre_tone_on_next
    """

    df = tone_air_window_df.copy()

    if trial_col not in df.columns:
        raise ValueError(f"{trial_col} not found in dataframe.")

    current_epochs = [
        "pre_tone_on",
        "post_tone_on",
        "pre_air_on",
        "post_air_on",
        "pre_tone_off",
        "post_tone_off",
        "pre_air_on_mid",
        "post_air_on_mid",
        "pre_air_off",
        "post_air_off",
        "pre_air_off_mid",
        "post_air_off_mid",
    ]

    # Current-trial epochs
    cur = df[df["tone_air_epoch_label"].isin(current_epochs)].copy()
    cur["cycle_trial"] = cur[trial_col].astype(int)
    cur["tone_air_cycle_epoch"] = cur["tone_air_epoch_label"]

    # Next pre-tone baseline:
    # pre_tone_on from trial N+1 becomes pre_tone_on_next for cycle N
    nxt = df[df["tone_air_epoch_label"] == "pre_tone_on"].copy()
    nxt["cycle_trial"] = nxt[trial_col].astype(int) - 1
    nxt["tone_air_cycle_epoch"] = "pre_tone_on_next"
    nxt = nxt[nxt["cycle_trial"] >= 1].copy()

    cycle_df = pd.concat([cur, nxt], ignore_index=True)

    # IDs
    cycle_df["animal_day"] = (
        cycle_df["animal"].astype(str) + ":" +
        cycle_df["date"].astype(str)
    )

    cycle_df["cycle_id"] = (
        cycle_df["animal"].astype(str) + ":" +
        cycle_df["date"].astype(str) + ":" +
        cycle_df["cycle_trial"].astype(str)
    )

    # Cycle-level time/session reference = pre_tone_on of that cycle
    cycle_info = (
        cycle_df[cycle_df["tone_air_cycle_epoch"] == "pre_tone_on"]
        [["animal", "date", "cycle_trial", "session_time_min", "tone_air_session"]]
        .drop_duplicates()
        .rename(columns={
            "session_time_min": "cycle_session_time_min",
            "tone_air_session": "cycle_tone_air_session",
        })
    )

    cycle_df = cycle_df.merge(
        cycle_info,
        on=["animal", "date", "cycle_trial"],
        how="left"
    )

    # Optional: keep only complete cycles
    if require_complete_cycles:
        counts = (
            cycle_df
            .groupby("cycle_id")["tone_air_cycle_epoch"]
            .nunique()
        )

        complete_cycle_ids = counts[counts == len(tone_air_cycle_order)].index

        cycle_df = cycle_df[
            cycle_df["cycle_id"].isin(complete_cycle_ids)
        ].copy()

    # Ordered categorical
    cycle_df["tone_air_cycle_epoch"] = pd.Categorical(
        cycle_df["tone_air_cycle_epoch"],
        categories=tone_air_cycle_order,
        ordered=True,
    )

    # Center predictors
    cycle_df["tone_air_session_c"] = (
        cycle_df["cycle_tone_air_session"] -
        cycle_df["cycle_tone_air_session"].mean()
    )

    cycle_df["cycle_session_10m_c"] = (
        cycle_df["cycle_session_time_min"] -
        cycle_df["cycle_session_time_min"].mean()
    ) / 10.0

    return cycle_df.reset_index(drop=True)


tone_air_cycle_df = make_tone_air_cycle_df(
    tone_air_window_df,
    trial_col="event_number",
    require_complete_cycles=True,
)

print(tone_air_cycle_df.shape)

display(
    tone_air_cycle_df["tone_air_cycle_epoch"]
    .value_counts()
    .sort_index()
)

(21762, 70)


tone_air_cycle_epoch
pre_tone_on         1674
post_tone_on        1674
pre_air_on          1674
post_air_on         1674
pre_tone_off        1674
post_tone_off       1674
pre_air_on_mid      1674
post_air_on_mid     1674
pre_air_off         1674
post_air_off        1674
pre_air_off_mid     1674
post_air_off_mid    1674
pre_tone_on_next    1674
Name: count, dtype: int64

In [6]:
# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

print("Animals:", tone_air_cycle_df["animal"].nunique())
print("Animal-days:", tone_air_cycle_df["animal_day"].nunique())
print("Cycles:", tone_air_cycle_df["cycle_id"].nunique())
print("Observations:", len(tone_air_cycle_df))

display(
    tone_air_cycle_df[
        [
            "animal",
            "date",
            "cycle_trial",
            "tone_air_cycle_epoch",
            anchor_col,
            "window_position",
            "event_number",
            "cycle_tone_air_session",
            "cycle_session_time_min",
        ]
    ].head(40)
)

display(
    tone_air_cycle_df
    .groupby(["animal", "date"])["cycle_id"]
    .nunique()
    .reset_index(name="n_cycles")
    .head()
)

Animals: 5
Animal-days: 49
Cycles: 1674
Observations: 21762


,animal,date,cycle_trial,tone_air_cycle_epoch,anchor_name,window_position,event_number,cycle_tone_air_session,cycle_session_time_min
0,NML_04,2026_01_28,1,post_air_off_mid,air_off_mid,post,1,1.0,1.398697
1,NML_04,2026_01_28,1,pre_air_off_mid,air_off_mid,pre,1,1.0,1.398697
2,NML_04,2026_01_28,1,post_air_off,air_off,post,1,1.0,1.398697
3,NML_04,2026_01_28,1,pre_air_off,air_off,pre,1,1.0,1.398697
4,NML_04,2026_01_28,1,post_air_on_mid,air_on_mid,post,1,1.0,1.398697
5,NML_04,2026_01_28,1,pre_air_on_mid,air_on_mid,pre,1,1.0,1.398697
6,NML_04,2026_01_28,1,post_air_on,air_on,post,1,1.0,1.398697
7,NML_04,2026_01_28,1,pre_air_on,air_on,pre,1,1.0,1.398697
8,NML_04,2026_01_28,1,post_tone_off,tone_off,post,1,1.0,1.398697
9,NML_04,2026_01_28,1,pre_tone_off,tone_off,pre,1,1.0,1.398697


,animal,date,n_cycles
0,NML_04,2026_01_28,38
1,NML_04,2026_01_29,36
2,NML_04,2026_01_30,38
3,NML_04,2026_01_31,31
4,NML_04,2026_02_01,30


In [7]:
%load_ext rpy2.ipython

In [8]:
%%R -i tone_air_cycle_df -o tone_air_emmeans_R -o tone_air_contrasts_R -o tone_air_r2_R -o tone_air_fixed_R -o tone_air_session_trends_R -o tone_air_session_time_trends_R -o tone_air_model_info_R

library(lme4)
library(lmerTest)
library(emmeans)
library(broom.mixed)
library(dplyr)
library(performance)

# Use asymptotic df for emmeans because models are large
emm_options(lmer.df = "asymptotic")

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

tone_air_cycle_df$animal <- factor(tone_air_cycle_df$animal)
tone_air_cycle_df$animal_day <- factor(tone_air_cycle_df$animal_day)
tone_air_cycle_df$cycle_id <- factor(tone_air_cycle_df$cycle_id)

tone_air_cycle_df$tone_air_cycle_epoch <- factor(
  as.character(tone_air_cycle_df$tone_air_cycle_epoch),
  levels = c(
    "pre_tone_on",
    "post_tone_on",
    "pre_air_on",
    "post_air_on",
    "pre_tone_off",
    "post_tone_off",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_tone_on_next"
  ),
  ordered = FALSE
)

# --------------------------------------------------
# Outcomes
# --------------------------------------------------

tone_air_outcomes_R <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms",
  "mean_speed_net_cms",
  "distance_path_cm",
  "distance_net_cm"
)

tone_air_outcomes_R <- tone_air_outcomes_R[
  tone_air_outcomes_R %in% names(tone_air_cycle_df)
]

print(tone_air_outcomes_R)

# --------------------------------------------------
# Planned adjacent contrasts
# 13 epochs, so each contrast vector has length 13
# --------------------------------------------------

tone_air_contrast_list <- list(
  tone_onset_transition =
    c(-1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),

  tone_to_air_anticipatory_interval =
    c(0, -1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),

  air_onset_transition =
    c(0, 0, -1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0),

  early_air_on_to_tone_offset =
    c(0, 0, 0, -1, 1, 0, 0, 0, 0, 0, 0, 0, 0),

  tone_offset_transition =
    c(0, 0, 0, 0, -1, 1, 0, 0, 0, 0, 0, 0, 0),

  post_tone_offset_air_on_progression =
    c(0, 0, 0, 0, 0, -1, 1, 0, 0, 0, 0, 0, 0),

  air_on_mid_transition =
    c(0, 0, 0, 0, 0, 0, -1, 1, 0, 0, 0, 0, 0),

  late_air_on_progression =
    c(0, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0, 0, 0),

  air_offset_transition =
    c(0, 0, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0, 0),

  early_air_off_recovery =
    c(0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0),

  air_off_mid_transition =
    c(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1, 0),

  late_air_off_recovery_to_next_baseline =
    c(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1)
)

# --------------------------------------------------
# Fit one outcome and extract tables
# --------------------------------------------------

fit_tone_air_cycle_outcome <- function(outcome_name) {

  cat("\n\n==============================\n")
  cat("Fitting outcome:", outcome_name, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome_name,
    " ~ tone_air_cycle_epoch * tone_air_session_c * cycle_session_10m_c + ",
    "(1 | animal) + (1 | animal_day) + (1 | cycle_id)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = tone_air_cycle_df,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  model_info <- data.frame(
    outcome = outcome_name,
    n_obs = nobs(m),
    n_animals = nlevels(tone_air_cycle_df$animal),
    n_animal_day = nlevels(tone_air_cycle_df$animal_day),
    n_cycle_id = nlevels(tone_air_cycle_df$cycle_id),
    AIC = AIC(m),
    BIC = BIC(m),
    logLik = as.numeric(logLik(m)),
    singular = isSingular(m),
    stringsAsFactors = FALSE
  )

  emm <- emmeans(
    m,
    ~ tone_air_cycle_epoch,
    at = list(
      tone_air_session_c = 0,
      cycle_session_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  emm_df <- as.data.frame(emm)
  emm_df$outcome <- outcome_name

  contrast_df <- as.data.frame(
    contrast(
      emm,
      tone_air_contrast_list,
      adjust = "none"
    )
  )
  contrast_df$outcome <- outcome_name

  r2_obj <- performance::r2_nakagawa(m)

  r2_df <- data.frame(
    outcome = outcome_name,
    R2_marginal = r2_obj$R2_marginal,
    R2_conditional = r2_obj$R2_conditional,
    stringsAsFactors = FALSE
  )

  fixed_df <- broom.mixed::tidy(
    m,
    effects = "fixed",
    conf.int = TRUE
  )
  fixed_df$outcome <- outcome_name

  session_trends <- emtrends(
    m,
    ~ tone_air_cycle_epoch,
    var = "tone_air_session_c",
    at = list(
      cycle_session_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_trends_df <- as.data.frame(session_trends)
  session_trends_df$outcome <- outcome_name

  session_time_trends <- emtrends(
    m,
    ~ tone_air_cycle_epoch,
    var = "cycle_session_10m_c",
    at = list(
      tone_air_session_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_time_trends_df <- as.data.frame(session_time_trends)
  session_time_trends_df$outcome <- outcome_name

  return(
    list(
      model = m,
      model_info = model_info,
      emmeans = emm_df,
      contrasts = contrast_df,
      r2 = r2_df,
      fixed = fixed_df,
      session_trends = session_trends_df,
      session_time_trends = session_time_trends_df
    )
  )
}

# --------------------------------------------------
# Run all models
# --------------------------------------------------

tone_air_models_R <- list()
model_info_list <- list()
emmeans_list <- list()
contrasts_list <- list()
r2_list <- list()
fixed_list <- list()
session_trends_list <- list()
session_time_trends_list <- list()

for (outcome_name in tone_air_outcomes_R) {

  result <- fit_tone_air_cycle_outcome(outcome_name)

  tone_air_models_R[[outcome_name]] <- result$model
  model_info_list[[outcome_name]] <- result$model_info
  emmeans_list[[outcome_name]] <- result$emmeans
  contrasts_list[[outcome_name]] <- result$contrasts
  r2_list[[outcome_name]] <- result$r2
  fixed_list[[outcome_name]] <- result$fixed
  session_trends_list[[outcome_name]] <- result$session_trends
  session_time_trends_list[[outcome_name]] <- result$session_time_trends
}

# --------------------------------------------------
# Combine output tables
# --------------------------------------------------

tone_air_model_info_R <- bind_rows(model_info_list)
tone_air_emmeans_R <- bind_rows(emmeans_list)
tone_air_contrasts_R <- bind_rows(contrasts_list)
tone_air_r2_R <- bind_rows(r2_list)
tone_air_fixed_R <- bind_rows(fixed_list)
tone_air_session_trends_R <- bind_rows(session_trends_list)
tone_air_session_time_trends_R <- bind_rows(session_time_trends_list)

# --------------------------------------------------
# Reorder columns
# --------------------------------------------------

tone_air_emmeans_R <- tone_air_emmeans_R %>%
  relocate(outcome)

tone_air_contrasts_R <- tone_air_contrasts_R %>%
  relocate(outcome) %>%
  select(outcome, contrast, estimate, SE, df, z.ratio, p.value)

tone_air_fixed_R <- tone_air_fixed_R %>%
  relocate(outcome)

tone_air_session_trends_R <- tone_air_session_trends_R %>%
  relocate(outcome)

tone_air_session_time_trends_R <- tone_air_session_time_trends_R %>%
  relocate(outcome)

# --------------------------------------------------
# Print summaries
# --------------------------------------------------

cat("\n\n=== Tone-air model info ===\n")
print(tone_air_model_info_R)

cat("\n\n=== Tone-air adjacent contrasts ===\n")
print(tone_air_contrasts_R)

cat("\n\n=== Tone-air R2 ===\n")
print(tone_air_r2_R)

cat("\n\n=== Tone-air exposure-session trends by epoch ===\n")
print(tone_air_session_trends_R)

cat("\n\n=== Tone-air within-session-time trends by epoch ===\n")
print(tone_air_session_time_trends_R)

R[write to console]: Loading required package: Matrix

R[write to console]: 
Attaching package: ‘lmerTest’


R[write to console]: The following object is masked from ‘package:lme4’:

    lmer


R[write to console]: The following object is masked from ‘package:stats’:

    step


R[write to console]: Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'

R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] "frac_moving"         "frac_forward"        "mean_speed_path_cms"
[4] "mean_speed_net_cms"  "distance_path_cm"    "distance_net_cm"    


Fitting outcome: frac_moving 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: frac_forward 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions



...



Fitting outcome: distance_path_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: distance_net_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





=== Tone-air model info ===
              outcome n_obs n_animals n_animal_day n_cycle_id       AIC
1         frac_moving 21762         5           49       1674  4687.568
2        frac_forward 21762         5           49       1674  4238.129
3 mean_speed_path_cms 21762         5           49       1674 91855.438
4  mean_speed_net_cms 21762         5           49       1674 92756.242
5    distance_path_cm 21762         5           49       1674 91849.102
6     distance_net_cm 21762         5           49       1674 92781.938
        BIC     logLik singular
1  5134.892  -2287.784    FALSE
2  4685.453  -2063.065    FALSE
3 92302.762 -45871.719    FALSE
4 93203.565 -46322.121    FALSE
5 92296.425 -45868.551    FALSE
6 93229.262 -46334.969    FALSE


=== Tone-air adjacent contrasts ===
               outcome                               contrast     estimate
1          frac_moving                  tone_onset_transition  0.152895125
2          frac_moving      tone_to_air_anticipatory_i

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Choose outcomes
# --------------------------------------------------

tone_air_outcomes = [
    "frac_moving",
    "frac_forward",
    "mean_speed_path_cms",
    "mean_speed_net_cms",
    "distance_path_cm",
    "distance_net_cm",
]

tone_air_outcomes = [
    x for x in tone_air_outcomes
    if x in tone_air_cycle_df.columns
]

print(tone_air_outcomes)

# --------------------------------------------------
# Copy dataframe
# --------------------------------------------------

df = tone_air_cycle_df.copy()

# --------------------------------------------------
# Fixed session-time bins
# --------------------------------------------------
# These are based on absolute minutes within the session.

df["session_time_bin"] = pd.cut(
    df["cycle_session_time_min"],
    bins=[0, 6, 12, np.inf],
    labels=["early_0_6min", "middle_6_12min", "late_12plus_min"],
    right=False
)

print(df["session_time_bin"].value_counts(dropna=False))

# --------------------------------------------------
# Optional: create tone-air training bins
# --------------------------------------------------
# This is better for RM-ANOVA than using every session as a separate factor.

df["tone_air_training_bin"] = pd.qcut(
    df["cycle_tone_air_session"],
    q=3,
    labels=["early_training", "middle_training", "late_training"],
    duplicates="drop"
)

print(df["tone_air_training_bin"].value_counts(dropna=False))

# --------------------------------------------------
# Aggregate to animal × training bin × session-time bin × epoch
# --------------------------------------------------

group_cols_training_bin = [
    "animal",
    "tone_air_training_bin",
    "session_time_bin",
    "tone_air_cycle_epoch",
]

tone_air_rm_trainingbin_df = (
    df
    .dropna(subset=group_cols_training_bin)
    .groupby(group_cols_training_bin, observed=True)[tone_air_outcomes]
    .mean()
    .reset_index()
)

print(tone_air_rm_trainingbin_df.shape)
display(tone_air_rm_trainingbin_df.head())

# --------------------------------------------------
# Aggregate to animal × session/day × session-time bin × epoch
# --------------------------------------------------
# Use this if you really want day/session as a factor.

group_cols_day = [
    "animal",
    "cycle_tone_air_session",
    "session_time_bin",
    "tone_air_cycle_epoch",
]

tone_air_rm_day_df = (
    df
    .dropna(subset=group_cols_day)
    .groupby(group_cols_day, observed=True)[tone_air_outcomes]
    .mean()
    .reset_index()
)

tone_air_rm_day_df["tone_air_session_factor"] = (
    tone_air_rm_day_df["cycle_tone_air_session"]
    .astype(int)
    .astype(str)
)

print(tone_air_rm_day_df.shape)
display(tone_air_rm_day_df.head())

['frac_moving', 'frac_forward', 'mean_speed_path_cms', 'mean_speed_net_cms', 'distance_path_cm', 'distance_net_cm']
session_time_bin
late_12plus_min    7761
early_0_6min       7475
middle_6_12min     6526
Name: count, dtype: int64
tone_air_training_bin
early_training     9282
middle_training    6357
late_training      6123
Name: count, dtype: int64
(585, 10)


,animal,tone_air_training_bin,session_time_bin,tone_air_cycle_epoch,frac_moving,frac_forward,mean_speed_path_cms,mean_speed_net_cms,distance_path_cm,distance_net_cm
0,NML_04,early_training,early_0_6min,pre_tone_on,0.249893,0.183751,0.437058,0.284843,0.444570,0.283721
1,NML_04,early_training,early_0_6min,post_tone_on,0.338347,0.242347,0.597375,0.375043,0.609329,0.375874
2,NML_04,early_training,early_0_6min,pre_air_on,0.083764,0.053129,0.123882,0.041815,0.128456,0.040212
3,NML_04,early_training,early_0_6min,post_air_on,0.725689,0.533271,2.252345,1.842409,2.268649,1.843068
4,NML_04,early_training,early_0_6min,pre_tone_off,0.920098,0.871418,4.970649,4.848736,4.982985,4.850061


(1885, 11)


,animal,cycle_tone_air_session,session_time_bin,tone_air_cycle_epoch,frac_moving,frac_forward,mean_speed_path_cms,mean_speed_net_cms,distance_path_cm,distance_net_cm,tone_air_session_factor
0,NML_04,1.0,early_0_6min,pre_tone_on,0.336164,0.303491,0.510547,0.418204,0.514079,0.413548,1
1,NML_04,1.0,early_0_6min,post_tone_on,0.284455,0.246836,0.596798,0.457800,0.612325,0.461529,1
2,NML_04,1.0,early_0_6min,pre_air_on,0.070727,0.042364,0.116849,0.005470,0.118809,0.004570,1
3,NML_04,1.0,early_0_6min,post_air_on,0.674945,0.508127,2.069965,1.642781,2.081448,1.647337,1
4,NML_04,1.0,early_0_6min,pre_tone_off,0.890582,0.845200,4.618073,4.453741,4.622140,4.443926,1


In [10]:
%%R -i tone_air_rm_trainingbin_df -o tone_air_rm_anova_trainingbin_R

library(dplyr)

# Install afex if needed:
# install.packages("afex")

library(afex)
library(emmeans)

afex_options(type = 3)

tone_air_rm_trainingbin_df$animal <- factor(tone_air_rm_trainingbin_df$animal)

tone_air_rm_trainingbin_df$tone_air_cycle_epoch <- factor(
  as.character(tone_air_rm_trainingbin_df$tone_air_cycle_epoch),
  levels = c(
    "pre_tone_on",
    "post_tone_on",
    "pre_air_on",
    "post_air_on",
    "pre_tone_off",
    "post_tone_off",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_tone_on_next"
  )
)

tone_air_rm_trainingbin_df$session_time_bin <- factor(
  tone_air_rm_trainingbin_df$session_time_bin,
  levels = c("early_0_6min", "middle_6_12min", "late_12plus_min")
)

tone_air_rm_trainingbin_df$tone_air_training_bin <- factor(
  tone_air_rm_trainingbin_df$tone_air_training_bin,
  levels = c("early_training", "middle_training", "late_training")
)

outcomes <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms",
  "mean_speed_net_cms",
  "distance_path_cm",
  "distance_net_cm"
)

outcomes <- outcomes[outcomes %in% names(tone_air_rm_trainingbin_df)]

rm_results <- list()

for (outcome in outcomes) {

  cat("\n\n==============================\n")
  cat(outcome, "\n")
  cat("==============================\n")

  aov_model <- aov_ez(
    id = "animal",
    dv = outcome,
    data = tone_air_rm_trainingbin_df,
    within = c(
      "tone_air_cycle_epoch",
      "tone_air_training_bin",
      "session_time_bin"
    ),
    anova_table = list(correction = "GG", es = "pes")
  )

  print(aov_model)

  rm_results[[outcome]] <- aov_model
}

# Convert ANOVA tables to one dataframe
anova_tables <- list()

for (outcome in outcomes) {
  tmp <- as.data.frame(rm_results[[outcome]]$anova_table)
  tmp$effect <- rownames(tmp)
  tmp$outcome <- outcome
  rownames(tmp) <- NULL
  anova_tables[[outcome]] <- tmp
}

tone_air_rm_anova_trainingbin_R <- bind_rows(anova_tables)

tone_air_rm_anova_trainingbin_R <- tone_air_rm_anova_trainingbin_R %>%
  relocate(outcome, effect)

print(tone_air_rm_anova_trainingbin_R)

R[write to console]: ************
Welcome to afex. For support visit: http://afex.singmann.science/

R[write to console]: - Functions for ANOVAs: aov_car(), aov_ez(), and aov_4()
- Methods for calculating p-values with mixed(): 'S', 'KR', 'LRT', and 'PB'
- 'afex_aov' and 'mixed' objects can be passed to emmeans() for follow-up tests
- Get and set global package options with: afex_options()
- Set sum-to-zero contrasts globally: set_sum_contrasts()
- For example analyses see: browseVignettes("afex")
************

R[write to console]: 
Attaching package: ‘afex’


R[write to console]: The following object is masked from ‘package:lme4’:

    lmer






frac_moving 
Anova Table (Type 3 tests)

Response: frac_moving
                                                       Effect         df  MSE
1                                        tone_air_cycle_epoch     12, 48 0.14
2                                       tone_air_training_bin 1.13, 4.52 0.07
3                                            session_time_bin 1.17, 4.67 0.04
4                  tone_air_cycle_epoch:tone_air_training_bin     24, 96 0.01
5                       tone_air_cycle_epoch:session_time_bin     24, 96 0.01
6                      tone_air_training_bin:session_time_bin 2.27, 9.10 0.01
7 tone_air_cycle_epoch:tone_air_training_bin:session_time_bin    48, 192 0.00
          F  pes p.value
1 35.90 *** .900   <.001
2      1.26 .239    .328
3      1.28 .242    .325
4    1.62 + .288    .053
5  3.83 *** .489   <.001
6      0.96 .193    .431
7    1.58 * .283    .016
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


frac_fo

In [12]:
%%R

library(dplyr)

tone_air_session_trends_R <- tone_air_session_trends_R %>%
  mutate(
    z.ratio = tone_air_session_c.trend / SE,
    p.value = 2 * (1 - pnorm(abs(z.ratio)))
  )

tone_air_session_time_trends_R <- tone_air_session_time_trends_R %>%
  mutate(
    z.ratio = cycle_session_10m_c.trend / SE,
    p.value = 2 * (1 - pnorm(abs(z.ratio)))
  )

print(tone_air_session_trends_R)
print(tone_air_session_time_trends_R)

               outcome tone_air_cycle_epoch tone_air_session_c.trend
1          frac_moving          pre_tone_on            -0.0049840127
2          frac_moving         post_tone_on            -0.0046154881
3          frac_moving           pre_air_on            -0.0034072912
4          frac_moving          post_air_on            -0.0046265030
5          frac_moving         pre_tone_off            -0.0070670463
6          frac_moving        post_tone_off            -0.0037820847
7          frac_moving       pre_air_on_mid            -0.0055741618
8          frac_moving      post_air_on_mid            -0.0060990039
9          frac_moving          pre_air_off             0.0001378639
10         frac_moving         post_air_off            -0.0040889395
11         frac_moving      pre_air_off_mid             0.0064490172
12         frac_moving     post_air_off_mid             0.0017121424
13         frac_moving     pre_tone_on_next            -0.0046314622
14        frac_forward          pr

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Identify anchor column
# --------------------------------------------------

anchor_col = "anchor_name" if "anchor_name" in encoder_epoch_df.columns else "epoch_name"
print("Using anchor column:", anchor_col)

print("\nPhases:")
print(encoder_epoch_df["phase"].value_counts())

print("\nAnchors by phase:")
display(
    encoder_epoch_df
    .groupby(["phase", anchor_col])
    .size()
    .reset_index(name="n")
    .sort_values(["phase", anchor_col])
)

# --------------------------------------------------
# Detect air and tone-air phases
# --------------------------------------------------

phases = encoder_epoch_df["phase"].dropna().astype(str).unique().tolist()

air_phase_candidates = [
    p for p in phases
    if ("air" in p.lower()) and ("tone" not in p.lower())
]

tone_air_phase_candidates = [
    p for p in phases
    if ("tone" in p.lower()) and ("air" in p.lower())
]

print("Air phase candidates:", air_phase_candidates)
print("Tone-air phase candidates:", tone_air_phase_candidates)

air_phase = air_phase_candidates[0]
tone_air_phase = tone_air_phase_candidates[0]

print("Using air phase:", air_phase)
print("Using tone-air phase:", tone_air_phase)

# --------------------------------------------------
# Helper to find anchors
# --------------------------------------------------

def find_anchor(df, phase, candidates, anchor_col):
    available = (
        df.loc[df["phase"] == phase, anchor_col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for c in candidates:
        if c in available:
            return c

    raise ValueError(
        f"No anchor found for phase={phase}. "
        f"Tried {candidates}. Available anchors: {available}"
    )

tone_on_anchor = find_anchor(
    encoder_epoch_df,
    tone_air_phase,
    candidates=[
        "tone_on",
        "tone_onset",
        "tone_on_3s_before_air",
        "Tone_on",
        "Tone_ON",
    ],
    anchor_col=anchor_col,
)

pseudo_tone_on_anchor = find_anchor(
    encoder_epoch_df,
    air_phase,
    candidates=[
        "pseudo_tone_on",
        "pseudo_tone",
        "pseudo_tone_onset",
        "pseudo_tone_ON",
        "matched_pseudo_tone_on",
    ],
    anchor_col=anchor_col,
)

print("tone_on anchor:", tone_on_anchor)
print("pseudo_tone_on anchor:", pseudo_tone_on_anchor)

Using anchor column: anchor_name

Phases:
phase
air_training         42362
habituation          38237
tone_air_training    21262
Name: count, dtype: int64

Anchors by phase:


,phase,anchor_name,n
0,air_training,air_off,7060
1,air_training,air_off_mid,7033
2,air_training,air_on,7067
3,air_training,air_on_mid,7069
4,air_training,pseudo_tone_off,7070
5,air_training,pseudo_tone_on,7063
6,habituation,LED_off,6373
7,habituation,LED_off_mid,6327
8,habituation,LED_on,6390
9,habituation,LED_on_mid,6386


Air phase candidates: ['air_training']
Tone-air phase candidates: ['tone_air_training']
Using air phase: air_training
Using tone-air phase: tone_air_training
tone_on anchor: tone_on
pseudo_tone_on anchor: pseudo_tone_on


In [14]:
def prepare_cue_on_vs_pseudo_df(
    encoder_epoch_df,
    air_phase,
    tone_air_phase,
    pseudo_tone_on_anchor,
    tone_on_anchor,
    anchor_col,
    require_good_session=True,
    require_valid_window=True,
    require_complete_prepost=True,
):
    """
    Builds dataframe comparing:
        pseudo-tone-on during air training
        real tone-on during tone-air training

    One row = one pre or post window around cue/pseudo-cue onset.
    """

    df = encoder_epoch_df.copy()

    # --------------------------------------------------
    # Pseudo-tone-on from air training
    # --------------------------------------------------

    pseudo_df = df[
        (df["phase"] == air_phase) &
        (df[anchor_col].astype(str) == pseudo_tone_on_anchor) &
        (df["window_position"].isin(["pre", "post"]))
    ].copy()

    pseudo_df["cue_type"] = "pseudo_tone_on"

    # --------------------------------------------------
    # Real tone-on from tone-air training
    # --------------------------------------------------

    tone_df = df[
        (df["phase"] == tone_air_phase) &
        (df[anchor_col].astype(str) == tone_on_anchor) &
        (df["window_position"].isin(["pre", "post"]))
    ].copy()

    tone_df["cue_type"] = "tone_on"

    # --------------------------------------------------
    # Combine
    # --------------------------------------------------

    cue_df = pd.concat([pseudo_df, tone_df], ignore_index=True)

    # --------------------------------------------------
    # Optional filters
    # --------------------------------------------------

    if require_good_session and "good_session_basic" in cue_df.columns:
        cue_df = cue_df[cue_df["good_session_basic"] == True].copy()

    if require_valid_window and "valid_window" in cue_df.columns:
        cue_df = cue_df[cue_df["valid_window"] == True].copy()

    # --------------------------------------------------
    # IDs
    # --------------------------------------------------

    cue_df["animal_day"] = (
        cue_df["animal"].astype(str) + ":" +
        cue_df["date"].astype(str)
    )

    trial_col = "event_number"

    if trial_col not in cue_df.columns:
        raise ValueError("event_number column not found.")

    cue_df["cue_event_id"] = (
        cue_df["cue_type"].astype(str) + ":" +
        cue_df["animal"].astype(str) + ":" +
        cue_df["date"].astype(str) + ":" +
        cue_df[trial_col].astype(str)
    )

    # --------------------------------------------------
    # Session time
    # --------------------------------------------------

    if "session_time_min" not in cue_df.columns:
        if "anchor_session_time_min" in cue_df.columns:
            cue_df["session_time_min"] = cue_df["anchor_session_time_min"]
        elif "anchor_time_s" in cue_df.columns:
            cue_df["session_time_min"] = cue_df["anchor_time_s"] / 60.0
        elif "session_time_s" in cue_df.columns:
            cue_df["session_time_min"] = cue_df["session_time_s"] / 60.0
        else:
            cue_df["session_time_min"] = np.nan

    # Centered and scaled in 10-min units
    cue_df["session_time_10m_c"] = (
        cue_df["session_time_min"] -
        cue_df["session_time_min"].mean()
    ) / 10.0

    # --------------------------------------------------
    # Phase/session number
    # --------------------------------------------------

    if "phase_session_number" in cue_df.columns:
        cue_df["phase_session"] = cue_df["phase_session_number"]
    elif "phase_day_number_good" in cue_df.columns:
        cue_df["phase_session"] = cue_df["phase_day_number_good"]
    elif "phase_day" in cue_df.columns:
        cue_df["phase_session"] = cue_df["phase_day"]
    else:
        raise ValueError(
            "Could not find phase session/day column. "
            "Expected phase_session_number, phase_day_number_good, or phase_day."
        )

    # Important:
    # Center session WITHIN cue type because pseudo-tone and real tone
    # come from different behavioral phases.
    cue_df["phase_session_c"] = (
        cue_df["phase_session"] -
        cue_df.groupby("cue_type")["phase_session"].transform("mean")
    )

    # --------------------------------------------------
    # Ordered factors
    # --------------------------------------------------

    cue_df["cue_type"] = pd.Categorical(
        cue_df["cue_type"],
        categories=["pseudo_tone_on", "tone_on"],
        ordered=True,
    )

    cue_df["window_position"] = pd.Categorical(
        cue_df["window_position"],
        categories=["pre", "post"],
        ordered=True,
    )

    # --------------------------------------------------
    # Keep only events that have both pre and post rows
    # --------------------------------------------------

    if require_complete_prepost:
        counts = (
            cue_df
            .groupby("cue_event_id")["window_position"]
            .nunique()
        )

        complete_event_ids = counts[counts == 2].index

        cue_df = cue_df[
            cue_df["cue_event_id"].isin(complete_event_ids)
        ].copy()

    return cue_df.reset_index(drop=True)


cue_on_df = prepare_cue_on_vs_pseudo_df(
    encoder_epoch_df=encoder_epoch_df,
    air_phase=air_phase,
    tone_air_phase=tone_air_phase,
    pseudo_tone_on_anchor=pseudo_tone_on_anchor,
    tone_on_anchor=tone_on_anchor,
    anchor_col=anchor_col,
)

print("cue_on_df shape:", cue_on_df.shape)

display(
    cue_on_df
    .groupby(["cue_type", "window_position"])
    .size()
    .reset_index(name="n_windows")
)

display(
    cue_on_df
    .groupby(["cue_type", "animal"])["cue_event_id"]
    .nunique()
    .reset_index(name="n_events")
)

display(
    cue_on_df[
        [
            "animal",
            "date",
            "phase",
            "cue_type",
            anchor_col,
            "event_number",
            "window_position",
            "phase_session",
            "phase_session_c",
            "session_time_min",
            "session_time_10m_c",
            "cue_event_id",
        ]
    ].head(20)
)

cue_on_df shape: (10594, 66)


,cue_type,window_position,n_windows
0,pseudo_tone_on,pre,3531
1,pseudo_tone_on,post,3531
2,tone_on,pre,1766
3,tone_on,post,1766


,cue_type,animal,n_events
0,pseudo_tone_on,NML_04,800
1,pseudo_tone_on,NML_05,440
2,pseudo_tone_on,NML_06,634
3,pseudo_tone_on,NML_07,813
4,pseudo_tone_on,NML_08,844
5,tone_on,NML_04,307
6,tone_on,NML_05,180
7,tone_on,NML_06,446
8,tone_on,NML_07,355
9,tone_on,NML_08,478


,animal,date,phase,cue_type,anchor_name,event_number,window_position,phase_session,phase_session_c,session_time_min,session_time_10m_c,cue_event_id
0,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,1,post,1,-7.118788,0.282607,-0.976135,pseudo_tone_on:NML_04:2026_01_12:1
1,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,1,pre,1,-7.118788,0.282607,-0.976135,pseudo_tone_on:NML_04:2026_01_12:1
2,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,2,post,1,-7.118788,0.601090,-0.944287,pseudo_tone_on:NML_04:2026_01_12:2
3,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,2,pre,1,-7.118788,0.601090,-0.944287,pseudo_tone_on:NML_04:2026_01_12:2
4,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,3,post,1,-7.118788,0.889413,-0.915455,pseudo_tone_on:NML_04:2026_01_12:3
5,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,3,pre,1,-7.118788,0.889413,-0.915455,pseudo_tone_on:NML_04:2026_01_12:3
6,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,4,post,1,-7.118788,1.171443,-0.887252,pseudo_tone_on:NML_04:2026_01_12:4
7,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,4,pre,1,-7.118788,1.171443,-0.887252,pseudo_tone_on:NML_04:2026_01_12:4
8,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,5,post,1,-7.118788,1.458487,-0.858547,pseudo_tone_on:NML_04:2026_01_12:5
9,NML_04,2026_01_12,air_training,pseudo_tone_on,pseudo_tone_on,5,pre,1,-7.118788,1.458487,-0.858547,pseudo_tone_on:NML_04:2026_01_12:5


In [15]:
%%R -i cue_on_df -o cue_on_model_info_R -o cue_on_prepost_R -o cue_on_difference_of_differences_R -o cue_on_window_cue_diffs_R -o cue_on_phase_session_response_trends_R -o cue_on_session_time_response_trends_R -o cue_on_fixed_R -o cue_on_r2_R

library(lme4)
library(lmerTest)
library(emmeans)
library(broom.mixed)
library(dplyr)
library(performance)

emm_options(lmer.df = "asymptotic")

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

cue_on_df$animal <- factor(cue_on_df$animal)
cue_on_df$animal_day <- factor(cue_on_df$animal_day)
cue_on_df$cue_event_id <- factor(cue_on_df$cue_event_id)

cue_on_df$cue_type <- factor(
  as.character(cue_on_df$cue_type),
  levels = c("pseudo_tone_on", "tone_on")
)

cue_on_df$window_position <- factor(
  as.character(cue_on_df$window_position),
  levels = c("pre", "post")
)

# --------------------------------------------------
# Outcomes
# --------------------------------------------------

cue_outcomes_R <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms",
  "mean_speed_net_cms",
  "distance_path_cm",
  "distance_net_cm"
)

cue_outcomes_R <- cue_outcomes_R[cue_outcomes_R %in% names(cue_on_df)]

print(cue_outcomes_R)

# --------------------------------------------------
# Function to fit one outcome
# --------------------------------------------------

fit_cue_on_outcome <- function(outcome_name) {

  cat("\n\n==============================\n")
  cat("Fitting outcome:", outcome_name, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome_name,
    " ~ cue_type * window_position * phase_session_c + ",
    "cue_type * window_position * session_time_10m_c + ",
    "(1 | animal) + (1 | animal_day) + (1 | cue_event_id)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = cue_on_df,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  # --------------------------------------------------
  # Model info
  # --------------------------------------------------

  model_info <- data.frame(
    outcome = outcome_name,
    n_obs = nobs(m),
    n_animals = nlevels(cue_on_df$animal),
    n_animal_day = nlevels(cue_on_df$animal_day),
    n_events = nlevels(cue_on_df$cue_event_id),
    AIC = AIC(m),
    BIC = BIC(m),
    logLik = as.numeric(logLik(m)),
    singular = isSingular(m),
    stringsAsFactors = FALSE
  )

  # --------------------------------------------------
  # Fixed effects
  # --------------------------------------------------

  fixed_df <- broom.mixed::tidy(
    m,
    effects = "fixed",
    conf.int = TRUE
  )
  fixed_df$outcome <- outcome_name

  # --------------------------------------------------
  # Estimated marginal means at reference:
  # phase_session_c = 0
  # session_time_10m_c = 0
  # --------------------------------------------------

  emm <- emmeans(
    m,
    ~ window_position | cue_type,
    at = list(
      phase_session_c = 0,
      session_time_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  # Pre/post effect separately for pseudo-tone and real tone
  prepost <- as.data.frame(
    contrast(
      emm,
      method = list(post_minus_pre = c(-1, 1)),
      by = "cue_type",
      adjust = "none"
    )
  )
  prepost$outcome <- outcome_name

  # Difference-of-differences:
  # (tone post-pre) - (pseudo post-pre)
  did <- as.data.frame(
    contrast(
      contrast(
        emm,
        method = list(post_minus_pre = c(-1, 1)),
        by = "cue_type",
        adjust = "none"
      ),
      method = list(tone_minus_pseudo = c(-1, 1)),
      by = NULL,
      adjust = "none"
    )
  )
  did$outcome <- outcome_name

  # Tone vs pseudo separately at pre and post windows
  emm2 <- emmeans(
    m,
    ~ cue_type | window_position,
    at = list(
      phase_session_c = 0,
      session_time_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  cue_diffs <- as.data.frame(
    contrast(
      emm2,
      method = list(tone_minus_pseudo = c(-1, 1)),
      by = "window_position",
      adjust = "none"
    )
  )
  cue_diffs$outcome <- outcome_name

  # --------------------------------------------------
  # Does post-pre response change across phase session?
  # --------------------------------------------------

  trend_phase <- emtrends(
    m,
    ~ window_position | cue_type,
    var = "phase_session_c",
    at = list(
      session_time_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  phase_response_trends <- as.data.frame(
    contrast(
      trend_phase,
      method = list(post_minus_pre_slope = c(-1, 1)),
      by = "cue_type",
      adjust = "none"
    )
  )
  phase_response_trends$outcome <- outcome_name

  # Difference in phase-session response slope:
  # tone response slope - pseudo response slope
  phase_response_did <- as.data.frame(
    contrast(
      contrast(
        trend_phase,
        method = list(post_minus_pre_slope = c(-1, 1)),
        by = "cue_type",
        adjust = "none"
      ),
      method = list(tone_minus_pseudo_slope = c(-1, 1)),
      by = NULL,
      adjust = "none"
    )
  )
  phase_response_did$outcome <- outcome_name
  phase_response_did$cue_type <- "tone_minus_pseudo"

  phase_response_trends <- bind_rows(
    phase_response_trends,
    phase_response_did
  )

  # --------------------------------------------------
  # Does post-pre response change across within-session time?
  # --------------------------------------------------

  trend_time <- emtrends(
    m,
    ~ window_position | cue_type,
    var = "session_time_10m_c",
    at = list(
      phase_session_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_time_response_trends <- as.data.frame(
    contrast(
      trend_time,
      method = list(post_minus_pre_slope = c(-1, 1)),
      by = "cue_type",
      adjust = "none"
    )
  )
  session_time_response_trends$outcome <- outcome_name

  # Difference in within-session-time response slope:
  # tone response slope - pseudo response slope
  session_time_response_did <- as.data.frame(
    contrast(
      contrast(
        trend_time,
        method = list(post_minus_pre_slope = c(-1, 1)),
        by = "cue_type",
        adjust = "none"
      ),
      method = list(tone_minus_pseudo_slope = c(-1, 1)),
      by = NULL,
      adjust = "none"
    )
  )
  session_time_response_did$outcome <- outcome_name
  session_time_response_did$cue_type <- "tone_minus_pseudo"

  session_time_response_trends <- bind_rows(
    session_time_response_trends,
    session_time_response_did
  )

  # --------------------------------------------------
  # R2
  # --------------------------------------------------

  r2_obj <- performance::r2_nakagawa(m)

  r2_df <- data.frame(
    outcome = outcome_name,
    R2_marginal = r2_obj$R2_marginal,
    R2_conditional = r2_obj$R2_conditional,
    stringsAsFactors = FALSE
  )

  return(
    list(
      model = m,
      model_info = model_info,
      fixed = fixed_df,
      prepost = prepost,
      did = did,
      cue_diffs = cue_diffs,
      phase_response_trends = phase_response_trends,
      session_time_response_trends = session_time_response_trends,
      r2 = r2_df
    )
  )
}

# --------------------------------------------------
# Run all outcomes
# --------------------------------------------------

cue_on_models_R <- list()
model_info_list <- list()
fixed_list <- list()
prepost_list <- list()
did_list <- list()
cue_diffs_list <- list()
phase_response_trends_list <- list()
session_time_response_trends_list <- list()
r2_list <- list()

for (outcome_name in cue_outcomes_R) {

  result <- fit_cue_on_outcome(outcome_name)

  cue_on_models_R[[outcome_name]] <- result$model
  model_info_list[[outcome_name]] <- result$model_info
  fixed_list[[outcome_name]] <- result$fixed
  prepost_list[[outcome_name]] <- result$prepost
  did_list[[outcome_name]] <- result$did
  cue_diffs_list[[outcome_name]] <- result$cue_diffs
  phase_response_trends_list[[outcome_name]] <- result$phase_response_trends
  session_time_response_trends_list[[outcome_name]] <- result$session_time_response_trends
  r2_list[[outcome_name]] <- result$r2
}

# --------------------------------------------------
# Combine tables
# --------------------------------------------------

cue_on_model_info_R <- bind_rows(model_info_list) %>% relocate(outcome)
cue_on_fixed_R <- bind_rows(fixed_list) %>% relocate(outcome)
cue_on_prepost_R <- bind_rows(prepost_list) %>% relocate(outcome)
cue_on_difference_of_differences_R <- bind_rows(did_list) %>% relocate(outcome)
cue_on_window_cue_diffs_R <- bind_rows(cue_diffs_list) %>% relocate(outcome)
cue_on_phase_session_response_trends_R <- bind_rows(phase_response_trends_list) %>% relocate(outcome)
cue_on_session_time_response_trends_R <- bind_rows(session_time_response_trends_list) %>% relocate(outcome)
cue_on_r2_R <- bind_rows(r2_list) %>% relocate(outcome)

# --------------------------------------------------
# Print key outputs
# --------------------------------------------------

cat("\n\n=== Model info ===\n")
print(cue_on_model_info_R)

cat("\n\n=== Pre/post response within pseudo-tone and real tone ===\n")
print(cue_on_prepost_R)

cat("\n\n=== Difference-of-differences: tone response minus pseudo-tone response ===\n")
print(cue_on_difference_of_differences_R)

cat("\n\n=== Tone vs pseudo at pre and post windows ===\n")
print(cue_on_window_cue_diffs_R)

cat("\n\n=== Phase-session modulation of post-pre response ===\n")
print(cue_on_phase_session_response_trends_R)

cat("\n\n=== Within-session-time modulation of post-pre response ===\n")
print(cue_on_session_time_response_trends_R)

cat("\n\n=== R2 ===\n")
print(cue_on_r2_R)

[1] "frac_moving"         "frac_forward"        "mean_speed_path_cms"
[4] "mean_speed_net_cms"  "distance_path_cm"    "distance_net_cm"    


Fitting outcome: frac_moving 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: frac_forward 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: mean_speed_path_cms 


...



Fitting outcome: distance_path_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: distance_net_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





=== Model info ===
              outcome n_obs n_animals n_animal_day n_events       AIC       BIC
1         frac_moving 10594         5          127     5297  4617.945  4734.233
2        frac_forward 10594         5          127     5297  3433.768  3550.057
3 mean_speed_path_cms 10594         5          127     5297 38481.964 38598.252
4  mean_speed_net_cms 10594         5          127     5297 38975.321 39091.610
5    distance_path_cm 10594         5          127     5297 38499.889 38616.178
6     distance_net_cm 10594         5          127     5297 38999.196 39115.484
      logLik singular
1  -2292.972    FALSE
2  -1700.884    FALSE
3 -19224.982    FALSE
4 -19471.660    FALSE
5 -19233.945    FALSE
6 -19483.598    FALSE


=== Pre/post response within pseudo-tone and real tone ===
               outcome       contrast       cue_type    estimate          SE
1          frac_moving post_minus_pre pseudo_tone_on -0.01454643 0.004685846
2          frac_moving post_minus_pre        tone_

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Identify anchor column
# --------------------------------------------------

anchor_col = "anchor_name" if "anchor_name" in encoder_epoch_df.columns else "epoch_name"
print("Using anchor column:", anchor_col)

print("\nPhases:")
print(encoder_epoch_df["phase"].value_counts())

print("\nAnchors by phase:")
display(
    encoder_epoch_df
    .groupby(["phase", anchor_col])
    .size()
    .reset_index(name="n")
    .sort_values(["phase", anchor_col])
)

# --------------------------------------------------
# Detect tone-air phase
# --------------------------------------------------

phases = encoder_epoch_df["phase"].dropna().astype(str).unique().tolist()

tone_air_phase_candidates = [
    p for p in phases
    if ("tone" in p.lower()) and ("air" in p.lower())
]

print("Tone-air phase candidates:", tone_air_phase_candidates)

tone_air_phase = tone_air_phase_candidates[0]
print("Using tone-air phase:", tone_air_phase)

# --------------------------------------------------
# Find tone_on anchor
# --------------------------------------------------

def find_anchor(df, phase, candidates, anchor_col):
    available = (
        df.loc[df["phase"] == phase, anchor_col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for c in candidates:
        if c in available:
            return c

    raise ValueError(
        f"No anchor found for phase={phase}. "
        f"Tried {candidates}. Available anchors: {available}"
    )


tone_on_anchor = find_anchor(
    encoder_epoch_df,
    tone_air_phase,
    candidates=[
        "tone_on",
        "tone_onset",
        "tone_on_3s_before_air",
        "Tone_on",
        "Tone_ON",
    ],
    anchor_col=anchor_col,
)

print("tone_on anchor:", tone_on_anchor)

Using anchor column: anchor_name

Phases:
phase
air_training         42362
habituation          38237
tone_air_training    21262
Name: count, dtype: int64

Anchors by phase:


,phase,anchor_name,n
0,air_training,air_off,7060
1,air_training,air_off_mid,7033
2,air_training,air_on,7067
3,air_training,air_on_mid,7069
4,air_training,pseudo_tone_off,7070
5,air_training,pseudo_tone_on,7063
6,habituation,LED_off,6373
7,habituation,LED_off_mid,6327
8,habituation,LED_on,6390
9,habituation,LED_on_mid,6386


Tone-air phase candidates: ['tone_air_training']
Using tone-air phase: tone_air_training
tone_on anchor: tone_on


In [17]:
def prepare_tone_on_response_df(
    encoder_epoch_df,
    tone_air_phase,
    tone_on_anchor,
    anchor_col,
    require_good_session=True,
    require_valid_window=True,
    require_complete_prepost=True,
):
    """
    Builds tone-on-only dataframe from tone-air training.

    One row = one pre or post window around tone onset.
    """

    df = encoder_epoch_df.copy()

    tone_df = df[
        (df["phase"] == tone_air_phase) &
        (df[anchor_col].astype(str) == tone_on_anchor) &
        (df["window_position"].isin(["pre", "post"]))
    ].copy()

    # --------------------------------------------------
    # Optional filters
    # --------------------------------------------------

    if require_good_session and "good_session_basic" in tone_df.columns:
        tone_df = tone_df[tone_df["good_session_basic"] == True].copy()

    if require_valid_window and "valid_window" in tone_df.columns:
        tone_df = tone_df[tone_df["valid_window"] == True].copy()

    # --------------------------------------------------
    # Animal-day/session ID
    # --------------------------------------------------

    tone_df["animal_day"] = (
        tone_df["animal"].astype(str) + ":" +
        tone_df["date"].astype(str)
    )

    # --------------------------------------------------
    # Tone-event ID
    # --------------------------------------------------

    trial_col = "event_number"

    if trial_col not in tone_df.columns:
        raise ValueError("event_number column not found.")

    tone_df["tone_event_id"] = (
        tone_df["animal"].astype(str) + ":" +
        tone_df["date"].astype(str) + ":" +
        tone_df[trial_col].astype(str)
    )

    # --------------------------------------------------
    # Session time in minutes
    # --------------------------------------------------

    if "session_time_min" not in tone_df.columns:
        if "anchor_session_time_min" in tone_df.columns:
            tone_df["session_time_min"] = tone_df["anchor_session_time_min"]
        elif "anchor_time_s" in tone_df.columns:
            tone_df["session_time_min"] = tone_df["anchor_time_s"] / 60.0
        elif "session_time_s" in tone_df.columns:
            tone_df["session_time_min"] = tone_df["session_time_s"] / 60.0
        else:
            tone_df["session_time_min"] = np.nan

    # --------------------------------------------------
    # Tone-air session number
    # --------------------------------------------------

    if "phase_session_number" in tone_df.columns:
        tone_df["tone_air_session"] = tone_df["phase_session_number"]
    elif "phase_day_number_good" in tone_df.columns:
        tone_df["tone_air_session"] = tone_df["phase_day_number_good"]
    elif "phase_day" in tone_df.columns:
        tone_df["tone_air_session"] = tone_df["phase_day"]
    else:
        raise ValueError(
            "Could not find phase session/day column. "
            "Expected phase_session_number, phase_day_number_good, or phase_day."
        )

    # --------------------------------------------------
    # Center predictors
    # --------------------------------------------------

    tone_df["tone_air_session_c"] = (
        tone_df["tone_air_session"] -
        tone_df["tone_air_session"].mean()
    )

    tone_df["session_time_10m_c"] = (
        tone_df["session_time_min"] -
        tone_df["session_time_min"].mean()
    ) / 10.0

    # --------------------------------------------------
    # Ordered factor
    # --------------------------------------------------

    tone_df["window_position"] = pd.Categorical(
        tone_df["window_position"],
        categories=["pre", "post"],
        ordered=True,
    )

    # --------------------------------------------------
    # Keep only events with both pre and post windows
    # --------------------------------------------------

    if require_complete_prepost:
        counts = (
            tone_df
            .groupby("tone_event_id")["window_position"]
            .nunique()
        )

        complete_event_ids = counts[counts == 2].index

        tone_df = tone_df[
            tone_df["tone_event_id"].isin(complete_event_ids)
        ].copy()

    return tone_df.reset_index(drop=True)


tone_on_response_df = prepare_tone_on_response_df(
    encoder_epoch_df=encoder_epoch_df,
    tone_air_phase=tone_air_phase,
    tone_on_anchor=tone_on_anchor,
    anchor_col=anchor_col,
)

print("tone_on_response_df shape:", tone_on_response_df.shape)

display(
    tone_on_response_df
    .groupby(["window_position"])
    .size()
    .reset_index(name="n_windows")
)

display(
    tone_on_response_df
    .groupby(["animal"])["tone_event_id"]
    .nunique()
    .reset_index(name="n_tone_events")
)

display(
    tone_on_response_df[
        [
            "animal",
            "date",
            "phase",
            anchor_col,
            "event_number",
            "window_position",
            "tone_air_session",
            "tone_air_session_c",
            "session_time_min",
            "session_time_10m_c",
            "tone_event_id",
        ]
    ].head(20)
)

tone_on_response_df shape: (3532, 65)


,window_position,n_windows
0,pre,1766
1,post,1766


,animal,n_tone_events
0,NML_04,307
1,NML_05,180
2,NML_06,446
3,NML_07,355
4,NML_08,478


,animal,date,phase,anchor_name,event_number,window_position,tone_air_session,tone_air_session_c,session_time_min,session_time_10m_c,tone_event_id
0,NML_04,2026_01_28,tone_air_training,tone_on,0,post,1,-4.345413,0.666360,-0.882629,NML_04:2026_01_28:0
1,NML_04,2026_01_28,tone_air_training,tone_on,0,pre,1,-4.345413,0.666360,-0.882629,NML_04:2026_01_28:0
2,NML_04,2026_01_28,tone_air_training,tone_on,1,post,1,-4.345413,1.398697,-0.809395,NML_04:2026_01_28:1
3,NML_04,2026_01_28,tone_air_training,tone_on,1,pre,1,-4.345413,1.398697,-0.809395,NML_04:2026_01_28:1
4,NML_04,2026_01_28,tone_air_training,tone_on,2,post,1,-4.345413,1.929357,-0.756329,NML_04:2026_01_28:2
5,NML_04,2026_01_28,tone_air_training,tone_on,2,pre,1,-4.345413,1.929357,-0.756329,NML_04:2026_01_28:2
6,NML_04,2026_01_28,tone_air_training,tone_on,3,post,1,-4.345413,2.285977,-0.720667,NML_04:2026_01_28:3
7,NML_04,2026_01_28,tone_air_training,tone_on,3,pre,1,-4.345413,2.285977,-0.720667,NML_04:2026_01_28:3
8,NML_04,2026_01_28,tone_air_training,tone_on,4,post,1,-4.345413,2.649487,-0.684316,NML_04:2026_01_28:4
9,NML_04,2026_01_28,tone_air_training,tone_on,4,pre,1,-4.345413,2.649487,-0.684316,NML_04:2026_01_28:4


In [18]:
%%R -i tone_on_response_df -o tone_on_model_info_R -o tone_on_prepost_R -o tone_on_session_response_trends_R -o tone_on_session_time_response_trends_R -o tone_on_response_grid_R -o tone_on_fixed_R -o tone_on_r2_R

library(lme4)
library(lmerTest)
library(emmeans)
library(broom.mixed)
library(dplyr)
library(performance)

emm_options(lmer.df = "asymptotic")

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

tone_on_response_df$animal <- factor(tone_on_response_df$animal)
tone_on_response_df$animal_day <- factor(tone_on_response_df$animal_day)
tone_on_response_df$tone_event_id <- factor(tone_on_response_df$tone_event_id)

tone_on_response_df$window_position <- factor(
  as.character(tone_on_response_df$window_position),
  levels = c("pre", "post")
)

# --------------------------------------------------
# Outcomes
# --------------------------------------------------

tone_outcomes_R <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms",
  "mean_speed_net_cms",
  "distance_path_cm",
  "distance_net_cm"
)

tone_outcomes_R <- tone_outcomes_R[tone_outcomes_R %in% names(tone_on_response_df)]

print(tone_outcomes_R)

# --------------------------------------------------
# Reference values for response grid
# --------------------------------------------------

session_sd <- sd(tone_on_response_df$tone_air_session_c, na.rm = TRUE)
time_sd <- sd(tone_on_response_df$session_time_10m_c, na.rm = TRUE)

session_grid <- c(-session_sd, 0, session_sd)
time_grid <- c(-time_sd, 0, time_sd)

session_grid_labels <- c("early_training", "mean_training", "late_training")
time_grid_labels <- c("early_session", "mean_session", "late_session")

# --------------------------------------------------
# Function to fit one outcome
# --------------------------------------------------

fit_tone_on_outcome <- function(outcome_name) {

  cat("\n\n==============================\n")
  cat("Fitting outcome:", outcome_name, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome_name,
    " ~ window_position * tone_air_session_c * session_time_10m_c + ",
    "(1 | animal) + (1 | animal_day) + (1 | tone_event_id)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = tone_on_response_df,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  # --------------------------------------------------
  # Model info
  # --------------------------------------------------

  model_info <- data.frame(
    outcome = outcome_name,
    n_obs = nobs(m),
    n_animals = nlevels(tone_on_response_df$animal),
    n_animal_day = nlevels(tone_on_response_df$animal_day),
    n_events = nlevels(tone_on_response_df$tone_event_id),
    AIC = AIC(m),
    BIC = BIC(m),
    logLik = as.numeric(logLik(m)),
    singular = isSingular(m),
    stringsAsFactors = FALSE
  )

  # --------------------------------------------------
  # Fixed effects
  # --------------------------------------------------

  fixed_df <- broom.mixed::tidy(
    m,
    effects = "fixed",
    conf.int = TRUE
  )

  fixed_df$outcome <- outcome_name

  # --------------------------------------------------
  # Tone response at mean training session and mean session time
  # --------------------------------------------------

  emm_ref <- emmeans(
    m,
    ~ window_position,
    at = list(
      tone_air_session_c = 0,
      session_time_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  prepost_ref <- as.data.frame(
    contrast(
      emm_ref,
      method = list(post_minus_pre = c(-1, 1)),
      adjust = "none"
    )
  )

  prepost_ref$outcome <- outcome_name

  # --------------------------------------------------
  # Does the tone response change across tone-air sessions?
  # This is the post-pre contrast of the session slope.
  # --------------------------------------------------

  session_trends <- emtrends(
    m,
    ~ window_position,
    var = "tone_air_session_c",
    at = list(
      session_time_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_response_trend <- as.data.frame(
    contrast(
      session_trends,
      method = list(post_minus_pre_session_slope = c(-1, 1)),
      adjust = "none"
    )
  )

  session_response_trend$outcome <- outcome_name

  # --------------------------------------------------
  # Does the tone response change across within-session time?
  # This is the post-pre contrast of the within-session-time slope.
  # --------------------------------------------------

  session_time_trends <- emtrends(
    m,
    ~ window_position,
    var = "session_time_10m_c",
    at = list(
      tone_air_session_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_time_response_trend <- as.data.frame(
    contrast(
      session_time_trends,
      method = list(post_minus_pre_session_time_slope = c(-1, 1)),
      adjust = "none"
    )
  )

  session_time_response_trend$outcome <- outcome_name

  # --------------------------------------------------
  # Response grid:
  # estimated post-pre response at early/mean/late training
  # and early/mean/late within-session time.
  # --------------------------------------------------

  emm_grid <- emmeans(
    m,
    ~ window_position | tone_air_session_c * session_time_10m_c,
    at = list(
      tone_air_session_c = session_grid,
      session_time_10m_c = time_grid
    ),
    lmer.df = "asymptotic"
  )

  response_grid <- as.data.frame(
    contrast(
      emm_grid,
      method = list(post_minus_pre = c(-1, 1)),
      by = c("tone_air_session_c", "session_time_10m_c"),
      adjust = "none"
    )
  )

  response_grid$outcome <- outcome_name

  # Add readable labels
  response_grid$training_level <- session_grid_labels[
    match(response_grid$tone_air_session_c, session_grid)
  ]

  response_grid$session_time_level <- time_grid_labels[
    match(response_grid$session_time_10m_c, time_grid)
  ]

  # --------------------------------------------------
  # R2
  # --------------------------------------------------

  r2_obj <- performance::r2_nakagawa(m)

  r2_df <- data.frame(
    outcome = outcome_name,
    R2_marginal = r2_obj$R2_marginal,
    R2_conditional = r2_obj$R2_conditional,
    stringsAsFactors = FALSE
  )

  return(
    list(
      model = m,
      model_info = model_info,
      fixed = fixed_df,
      prepost = prepost_ref,
      session_response_trend = session_response_trend,
      session_time_response_trend = session_time_response_trend,
      response_grid = response_grid,
      r2 = r2_df
    )
  )
}

# --------------------------------------------------
# Run all outcomes
# --------------------------------------------------

tone_on_models_R <- list()
model_info_list <- list()
fixed_list <- list()
prepost_list <- list()
session_response_trend_list <- list()
session_time_response_trend_list <- list()
response_grid_list <- list()
r2_list <- list()

for (outcome_name in tone_outcomes_R) {

  result <- fit_tone_on_outcome(outcome_name)

  tone_on_models_R[[outcome_name]] <- result$model
  model_info_list[[outcome_name]] <- result$model_info
  fixed_list[[outcome_name]] <- result$fixed
  prepost_list[[outcome_name]] <- result$prepost
  session_response_trend_list[[outcome_name]] <- result$session_response_trend
  session_time_response_trend_list[[outcome_name]] <- result$session_time_response_trend
  response_grid_list[[outcome_name]] <- result$response_grid
  r2_list[[outcome_name]] <- result$r2
}

# --------------------------------------------------
# Combine output tables
# --------------------------------------------------

tone_on_model_info_R <- bind_rows(model_info_list) %>% relocate(outcome)
tone_on_fixed_R <- bind_rows(fixed_list) %>% relocate(outcome)
tone_on_prepost_R <- bind_rows(prepost_list) %>% relocate(outcome)
tone_on_session_response_trends_R <- bind_rows(session_response_trend_list) %>% relocate(outcome)
tone_on_session_time_response_trends_R <- bind_rows(session_time_response_trend_list) %>% relocate(outcome)
tone_on_response_grid_R <- bind_rows(response_grid_list) %>% relocate(outcome)
tone_on_r2_R <- bind_rows(r2_list) %>% relocate(outcome)

# --------------------------------------------------
# Print key outputs
# --------------------------------------------------

cat("\n\n=== Model info ===\n")
print(tone_on_model_info_R)

cat("\n\n=== Tone pre/post response at mean training and mean session time ===\n")
print(tone_on_prepost_R)

cat("\n\n=== Does tone response change across tone-air sessions? ===\n")
print(tone_on_session_response_trends_R)

cat("\n\n=== Does tone response change across within-session time? ===\n")
print(tone_on_session_time_response_trends_R)

cat("\n\n=== Tone response grid: early/mean/late training × early/mean/late session ===\n")
print(tone_on_response_grid_R)

cat("\n\n=== Fixed effects ===\n")
print(tone_on_fixed_R)

cat("\n\n=== R2 ===\n")
print(tone_on_r2_R)

[1] "frac_moving"         "frac_forward"        "mean_speed_path_cms"
[4] "mean_speed_net_cms"  "distance_path_cm"    "distance_net_cm"    


Fitting outcome: frac_moving 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: frac_forward 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: mean_speed_path_cms 


...



Fitting outcome: distance_path_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: distance_net_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





=== Model info ===
              outcome n_obs n_animals n_animal_day n_events       AIC
1         frac_moving  3532         5           49     1766  1239.196
2        frac_forward  3532         5           49     1766   595.804
3 mean_speed_path_cms  3532         5           49     1766 10214.770
4  mean_speed_net_cms  3532         5           49     1766 10310.704
5    distance_path_cm  3532         5           49     1766 10228.594
6     distance_net_cm  3532         5           49     1766 10317.406
         BIC     logLik singular
1  1313.2317  -607.5981    FALSE
2   669.8395  -285.9020    FALSE
3 10288.8059 -5095.3852    FALSE
4 10384.7396 -5143.3521    FALSE
5 10302.6290 -5102.2968    FALSE
6 10391.4412 -5146.7029    FALSE


=== Tone pre/post response at mean training and mean session time ===
              outcome       contrast  estimate          SE  df   z.ratio
1         frac_moving post_minus_pre 0.1486416 0.007971113 Inf 18.647533
2        frac_forward post_minus_pre 0.1

In [20]:
# Check available columns
print("tone_on_fixed_R columns:")
print(tone_on_fixed_R.columns.tolist())

print("\ntone_on_response_grid_R columns:")
print(tone_on_response_grid_R.columns.tolist())

tone_on_fixed_R columns:
['outcome', 'effect', 'term', 'estimate', 'std.error', 'statistic', 'df', 'p.value', 'conf.low', 'conf.high']

tone_on_response_grid_R columns:
['outcome', 'contrast', 'tone_air_session_c', 'session_time_10m_c', 'estimate', 'SE', 'df', 'z.ratio', 'p.value', 'training_level', 'session_time_level']


In [21]:
import pandas as pd
import numpy as np

# Make a clean copy
fixed = tone_on_fixed_R.copy()

# Show all terms containing both tone_air_session_c and session_time_10m_c
session_x_time_terms = fixed[
    fixed["term"].str.contains("tone_air_session_c", regex=False) &
    fixed["term"].str.contains("session_time_10m_c", regex=False)
].copy()

display(
    session_x_time_terms[
        ["outcome", "term", "estimate", "std.error", "statistic", "p.value"]
    ].sort_values(["outcome", "term"])
)

,outcome,term,estimate,std.error,statistic,p.value
47,distance_net_cm,tone_air_session_c:session_time_10m_c,0.007304,0.015851,0.460804,0.644978
48,distance_net_cm,window_positionpost:tone_air_session_c:session...,-0.016188,0.014395,-1.124538,0.260938
39,distance_path_cm,tone_air_session_c:session_time_10m_c,0.010480,0.015710,0.667107,0.504764
40,distance_path_cm,window_positionpost:tone_air_session_c:session...,-0.020260,0.014121,-1.434773,0.151529
15,frac_forward,tone_air_session_c:session_time_10m_c,0.003492,0.003775,0.925093,0.354993
16,frac_forward,window_positionpost:tone_air_session_c:session...,-0.010904,0.004058,-2.687234,0.007272
7,frac_moving,tone_air_session_c:session_time_10m_c,0.005166,0.004078,1.266803,0.205322
8,frac_moving,window_positionpost:tone_air_session_c:session...,-0.012071,0.004603,-2.622573,0.008802
31,mean_speed_net_cms,tone_air_session_c:session_time_10m_c,0.007456,0.015845,0.470573,0.637985
32,mean_speed_net_cms,window_positionpost:tone_air_session_c:session...,-0.016441,0.014368,-1.144319,0.252647
